# Build Races Dimension

1. Read silver `races` table
1. Read silver `circuits` table
1. Join the data from `races` with `circuits` using `circuit_id`
1. Select the required columns
    - races.season 
    - races.round 
    - races.race_name 
    - races.race_date 
    - circuits.circuit_name 
    - circuits.locality 
    - circuits.country
1. Write the transformed data to gold `dim_races` table



#### Entity Relationship Diagram - Formula1 Silver Schema

![Formula1 Silver Data.png](../../z-course-images/formula1-silver-data-erd.png "Formula1 Silver Data.png")


#### Entity Relationship Diagram - Formula1 Gold Schema

![Formula1 Gold Data.png](../../z-course-images/formula1-gold-data-erd.png "Formula1 Gold Data.png")

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalogo}.{esquema_gold}.dim_carrera"

#### Step 1 - Read source tables
- `circuits`
- `races`

In [0]:
circuits_df = spark.table(f"{catalogo}.{esquema_silver}.circuitos")
races_df = spark.table(f"{catalogo}.{esquema_silver}.carreras")

#### Step 2 - Join `races` with `circuits` using `circuit_id`
Select the following columns  
  1. races.season 
  1. races.round 
  1. races.race_name 
  1. races.race_date 
  1. circuits.circuit_name 
  1. circuits.locality 
  1. circuits.country

In [0]:
dim_races_df = (
            races_df
                .join(
                    circuits_df,
                    races_df.id_circuito== circuits_df.id_circuito,
                    "inner"
                )
                .select (
                    races_df.temporada,
                    races_df.numero_carrera,
                    races_df.nombre_carrera,
                    races_df.fecha_carrera,
                    circuits_df.nombre_circuito,
                    circuits_df.localidad,
                    circuits_df.pais
                )
        )

In [0]:
display(dim_races_df)

#### Step 3 - Write the transformed data to the `gold` `dim_races` table

In [0]:
(
    dim_races_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))